# 深度學習框架應用（大學部）｜主題 4（上）：從一個神經元到一整個網路

這本筆記本配合投影片《主題 4（上）：全連接神經網路 MLP（多層感知器）》。投影片從頭到尾只用一個例子：用學測成績推薦科系的「推薦科系機」，同一位同學小明（國文 12、英文 13、數學 10 級分）。
五題都在同一台機器上做：先算一個系（資工系）的分數，再三個系一起算，再先算文科分、理科分，最後把分數換成機率、看正確答案扣分、往回調配分。只有練習三換一個小問題（樓梯間的兩個開關）。所有數字都是示範用的。**練習一～五，每題 20%，合計 100%**。

| 練習 | 題目 | 投影片（練習頁） | 觀念在哪一頁 |
|---|---|---|---|
| 一 | 一個神經元：資工系怎麼算小明（加權、門檻、ReLU、nn.Linear） | 第 67 頁 | 第 8、9、11 頁 |
| 二 | 一層：三個系一起算（矩陣乘法、形狀規則、nn.Linear(3, 3)） | 第 68 頁 | 第 20、22、23 頁 |
| 三 | 插曲：樓梯間的兩個開關 XOR，一層 vs 兩層 | 第 69 頁 | 第 32、33、34 頁 |
| 四 | 很多層：先算文理科，再算各系（中間要有 ReLU、參數量） | 第 70 頁 | 第 28、31、36、37 頁 |
| 五 | 機率、扣分、調配分（softmax、交叉熵、梯度、學習率） | 第 71 頁 | 第 42、45、53、56 頁 |

**怎麼用**

1. 用私人 Google 帳號登入（不使用學校帳號），按上方的「複製到雲端硬碟」（或「檔案 › 在雲端硬碟中儲存複本」）另存一份自己的複本，之後都改自己那份。**沒有另存就直接寫，分頁一關就全部不見。** 存好後先把檔名改成 `深度學習_W04上_學號_姓名.ipynb`（左上角點檔名就能改）。
2. 先跑「環境準備」，再由上往下做。每一題只有一格標 ✏️ 的程式格：把每個 `___` 換成答案再執行，提示就寫在同一行後面的註解。還沒把 `___` 換掉就執行，會出現紅字錯誤（例如 TypeError、AttributeError），錯誤訊息裡箭頭 `---->` 指的那一行就是還沒填的空格，填好再跑一次就好。
3. 練習三、四前面各有一格「準備」，直接執行，不用改。五題都是很小的張量，**CPU 就跑得動，不用開 GPU**；練習三要讓電腦自己練 1,000 次，大約幾秒鐘。改壞了就重新開原本的連結，再存一份複本。
4. **每題做完就截一張整個螢幕的圖（截圖 ①～⑤）**，要看得到身份指紋：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間，以及這一題的輸出。**看不到身份指紋，那一題不計分。**
5. 文字題裡的 **N 是你自己學號的末兩碼**（例如學號尾數 07，N＝07），每個人算出來的數字都不一樣。

**繳交**：做完照最後一格「繳交前檢查」，把**分享連結 1 個、.ipynb 檔 1 個、截圖 5 張（①～⑤）**交到 Flipclass 作業區，期限照 Flipclass 規定。

## 環境準備

五題都只用到 PyTorch，先跑這一格。最後一行讓 PyTorch 印數字時只印到小數第 3 位，比較好對答案。

In [ ]:
# ===== 環境準備 =====
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.set_printoptions(precision=3, sci_mode=False)  # 印到小數第 3 位
print('PyTorch：', torch.__version__)

## 練習一：一個神經元：資工系怎麼算小明 (20%)

> 題目在投影片**第 67 頁**｜觀念在第 8、9、11 頁

**推薦科系機上的哪一步**：第 ① 段。資工系的配分是國文 ×1、英文 ×1、數學 ×2，門檻 40 分。小明國英數 12、13、10 級分；小華 10、9、8 級分。

**這題在練**：一個神經元的四件事「乘上配分、加起來、減掉門檻、沒過門檻記 0」，再用 `nn.Linear(3, 1)` 設成同樣的配分和門檻對答案

**步驟**

1. 填 `.sum()`：每一科乘上配分再全部加起來，小明應該是 45，減掉門檻是 5
2. `nn.Linear(3, ___)` 填要算出幾個分數，再把門檻 b 抄進 `cs.bias`，兩位同學一起算出 [5, −5]
3. 填 ReLU：負的記 0，小明還是 5、小華變 0

**要繳交**

- 截圖 ①：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ✏️ 練習一：一個神經元（資工系怎麼算小明）
x = torch.tensor([12., 13., 10.])      # 小明：國英數
w = torch.tensor([1., 1., 2.])         # 資工系的配分
b = -40.                               # 門檻（偏置）
s = (x * w).___()                      # ✏️ 乘完全部加起來
print('加起來', s.item(), '減門檻', (s + b).item())
X = torch.tensor([[12., 13., 10.], [10., 9., 8.]])
cs = nn.Linear(3, ___)                 # ✏️ 算幾個分數
with torch.no_grad():
    cs.weight[:] = w                   # 抄配分
    cs.bias[:] = ___                   # ✏️ 抄門檻
z = cs(X).flatten()                    # 小明、小華
print('減門檻', z.tolist())
print('ReLU', torch.___(z).tolist())   # ✏️ 負的記 0

#### 📸 截圖 ①

跑完上面的 ✏️ 格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：三行結果（加起來與減門檻、nn.Linear 算出的兩位同學、ReLU 之後）

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. 資工系數學的配分是 2、國文和英文是 1，代表什麼？門檻 −40 為什麼不管誰來算都一樣？**

（在這裡寫）

**2. 小明的國文 12、英文 13 不變，數學改成「你學號末兩碼 N 的個位數＋5」。資工系最後的分數（ReLU 之後）是多少？寫出算式。**

（在這裡寫）

## 練習二：一層：三個系一起算 (20%)

> 題目在投影片**第 68 頁**｜觀念在第 20、22、23 頁

**推薦科系機上的哪一步**：第 ② 段。三個系各有一套配分：資工 [1, 1, 2]、外文 [1, 2, −1]、電機 [1, 1, 3]，門檻 40、20、52。小明、小華一起算。

**這題在練**：一層＝很多個神經元一起算＝一次矩陣乘法：把三套配分疊成 W（一列一個系，3×3），兩位同學疊成 X（一列一個人，2×3），用 `X @ W.T + b` 一次算完，並看懂形狀規則「內側相等、取外側」

**步驟**

1. 填 `W.T`：配分表轉置，同學的 3 科才對得上配分表的 3 科
2. 看印出來的形狀 (2, 3) @ (3, 3) → (2, 3)：2 位同學×3 個系；Z 應該是 [[5, 8, 3], [−5, 0, −9]]，ReLU 後小華全部是 0
3. 用 `nn.Linear(3, 3)` 設同樣的配分和門檻，結果一模一樣；注意它的 weight 形狀是 (3, 3)，一列一個系、一欄一科

**要繳交**

- 截圖 ②：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ✏️ 練習二：一層（三個系一起算）
W = torch.tensor([[1., 1., 2.],       # 資工
                  [1., 2., -1.],      # 外文
                  [1., 1., 3.]])      # 電機
b = torch.tensor([-40., -20., -52.])  # 三個門檻
X = torch.tensor([[12., 13., 10.], [10., 9., 8.]])
Z = X @ W.___ + b                     # ✏️ 轉置
print(X.shape, '@', W.T.shape, '→', Z.shape)
print('Z', Z.tolist(), 'ReLU', torch.relu(Z).tolist())
layer = nn.Linear(3, ___)             # ✏️ 幾個系
with torch.no_grad():
    layer.weight[:] = ___             # ✏️ 一列一個系
    layer.bias[:] = b
print('nn.Linear', layer(X).tolist(), layer.weight.shape)

#### 📸 截圖 ②

跑完上面的 ✏️ 格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：形狀那一行、Z 與 ReLU、nn.Linear 的結果和 weight 形狀

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. 為什麼要寫 W.T？用「內側相等、取外側」說明 (2, 3) @ (3, 3) → (2, 3)，並說出 Z 第 1 列第 2 格的 8 是誰、在哪一系的分數。**

（在這裡寫）

**2. 同學丙的國文 12、英文 13 級分，數學是「你學號末兩碼 N 的個位數＋6」。三個系的分數（ReLU 之前）各是多少？數學每多 1 級分，哪一系漲最多、哪一系反而變少？**

（在這裡寫）

## 練習三：插曲：樓梯間的兩個開關（XOR） (20%)

> 題目在投影片**第 69 頁**｜觀念在第 32、33、34 頁

**推薦科系機上的哪一步**：第 ③ 段的插曲，換一個小問題。樓梯間樓上、樓下各有一個開關：只有一個開關扳上去時燈亮；兩個都沒扳、兩個都扳上去，燈都是暗的。

**這題在練**：先用手設的配分解開 XOR（h₁＝扳上去幾個、h₂＝兩個都扳上去了嗎、燈＝h₁−2h₂）；再讓電腦自己練「一層」和「兩層（中間 16 個神經元）」，比一比

**步驟**

1. 先執行「準備」那一格：放好四種開關情況 X、燈亮不亮 y，以及讓電腦自己練的 `train_xor()`
2. 填 h₂ 要減的數和燈的算法，手設配分應該得到 [0, 1, 1, 0]
3. 填中間那一層的神經元數 16，看「一層」最多只到 75%、「兩層」到 100%

> `train_xor()` 用的是主題 4（下）才會細講的寫法：BCEWithLogitsLoss 算扣分、Adam 調配分、練 1,000 次。這題只要看結果。

**要繳交**

- 截圖 ③：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ===== 練習三：準備（直接執行，不用改）=====
X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])  # (樓下, 樓上)：1＝扳上去
y = torch.tensor([[0.], [1.], [1.], [0.]])  # 只扳一個，燈才亮


def train_xor(model, steps=1000, lr=0.05):
    """讓電腦自己調配分：用 Adam 練 steps 次，回傳 (準確率, 扣分)。"""
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()      # 亮或不亮：二選一的扣分方法
    for _ in range(steps):
        loss = loss_fn(model(X), y)       # 四種情況都看一次、算扣分
        opt.zero_grad()                   # 擦掉上一次的責任紀錄
        loss.backward()                   # 往回算每個配分的責任
        opt.step()                        # 照責任調配分
    with torch.no_grad():
        out = model(X)
        acc = ((out > 0).float() == y).float().mean().item()
        return acc, loss_fn(out, y).item()


print('兩個開關的四種情況：', X.tolist())
print('燈亮不亮：', y.flatten().tolist())

In [ ]:
# ✏️ 練習三：樓梯間的兩個開關（XOR）
s = X[:, 0] + X[:, 1]         # 扳上去幾個
h1 = torch.relu(s)
h2 = torch.relu(s - ___)      # ✏️ 兩個都扳上去了嗎
light = h1 - ___ * h2         # ✏️ 燈亮不亮
print('手設配分：', light.int().tolist())
torch.manual_seed(0)          # 固定亂數
one = nn.Linear(2, 1)         # 一層：一條直線
two = nn.Sequential(nn.Linear(2, ___), nn.ReLU(),  # ✏️
                    nn.Linear(16, 1))  # 兩層
for name, m in [('一層', one), ('兩層', two)]:
    acc, loss = train_xor(m)  # 電腦自己調配分
    print(f'{name}：準確率 {acc:.0%}，扣分 {loss:.3f}')

#### 📸 截圖 ③

跑完上面的 ✏️ 格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：手設配分的結果、一層和兩層的準確率與扣分

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. 為什麼「一層」怎麼練都到不了 100%，扣分停在 0.693 附近？**

（在這裡寫）

**2. 把 torch.manual_seed(0) 改成 torch.manual_seed(N)（N＝學號末兩碼）再跑一次，寫下兩種的準確率和扣分。為什麼中間放 16 個神經元，不只放手設的 2 個？**

（在這裡寫）

## 練習四：很多層：先算文理科，再算各系 (20%)

> 題目在投影片**第 70 頁**｜觀念在第 28、31、36、37 頁

**推薦科系機上的哪一步**：第 ③ 段。中間一層先算文科分＝ReLU(國＋英−20)、理科分＝ReLU(2×數−18)，最後一層再算三個系：資工＝理、外文＝文−2×理、電機＝理−2。

**這題在練**：算出小明的中間分數 h＝[5, 2] 和三個系的分數 [2, 1, 0]；親眼看到「中間沒有 ReLU」時兩層可以併成一層，中間有 ReLU 就併不起來；再用 `numel()` 數配分表一共有幾格（17 和 101,770）

**步驟**

1. 先執行「練習四、五：準備」那一格，放好兩層的配分 W1、b1、W2、b2，以及小明、小華的成績
2. 填 ReLU：小明的 h 應該是 [5, 2]、logits 是 [2, 1, 0]；填 `W1`，把兩層併成一層，小華「沒有 ReLU」和「一次加權」都是 [−2, 3, −4]，「有 ReLU」卻是 [0, 0, −2]
3. 填最後一層的 10 和 `numel`：推薦科系機 17 格，認數字的 784 → 128 → 10 是 101,770 格

**要繳交**

- 截圖 ④：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ===== 練習四、五：準備（直接執行，不用改）=====
# 推薦科系機的配分（跟投影片第 ③ 段一樣）
W1 = torch.tensor([[1., 1., 0.],     # 文科分＝國＋英
                   [0., 0., 2.]])    # 理科分＝2×數
b1 = torch.tensor([-20., -18.])      # 文科分、理科分的門檻
W2 = torch.tensor([[0., 1.],         # 資工＝理
                   [1., -2.],        # 外文＝文－2×理
                   [0., 1.]])        # 電機＝理－2
b2 = torch.tensor([0., 0., -2.])     # 三個系的門檻
xm = torch.tensor([12., 13., 10.])   # 小明的國英數
xh = torch.tensor([10., 9., 8.])     # 小華的國英數
T = torch.tensor(2)                  # 學長 C 的答案：電機（第 2 個）
print('W1', tuple(W1.shape), 'W2', tuple(W2.shape), '準備好了')

In [ ]:
# ✏️ 練習四：很多層（先算文理科，再算各系）
h = torch.___(W1 @ xm + b1)         # ✏️ 沒過記 0
print('h', h.tolist(), 'logits', (W2 @ h + b2).tolist())
zh = W1 @ xh + b1                   # 小華（還沒記 0）
print('沒有 ReLU', (W2 @ zh + b2).tolist())
Wm, bm = W2 @ ___, W2 @ b1 + b2     # ✏️ 併成一層
print('一次加權', (Wm @ xh + bm).tolist())
print('有 ReLU', (W2 @ zh.relu() + b2).tolist())
m = nn.Sequential(nn.Linear(3, 2), nn.ReLU(),
                  nn.Linear(2, 3))  # 推薦科系機
big = nn.Sequential(nn.Linear(784, 128), nn.ReLU(),
                    nn.Linear(128, ___))  # ✏️ 0～9
for net in (m, big):                # 數配分表
    print(sum(p.___() for p in net.parameters()))  # ✏️

#### 📸 截圖 ④

跑完上面的 ✏️ 格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：h 和 logits、小華的三行（沒有 ReLU、一次加權、有 ReLU）、兩個參數量

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. 小華「沒有 ReLU」和「一次加權」都是 [−2, 3, −4]，為什麼？「有 ReLU」為什麼變成 [0, 0, −2]，用併起來的那一套配分就算不出來？**

（在這裡寫）

**2. 用你的學號末兩碼 N：把認數字的網路改成 784 → (100＋N) → 10，手算參數量（寫出算式），再改程式驗證。**

（在這裡寫）

## 練習五：機率、扣分、調配分 (20%)

> 題目在投影片**第 71 頁**｜觀念在第 42、45、53、56 頁

**推薦科系機上的哪一步**：第 ④、⑤ 段。學長 C 的成績和小明一樣，中間分數也是 [5, 2]、三個系的分數 [2, 1, 0]；他後來讀電機（第 2 個），推薦機推薦錯了。

**這題在練**：自己寫 softmax（換成機率）和交叉熵（扣分）；用 `backward()` 往回算責任，看「門檻的梯度＝機率−答案」；再用學習率 0.1 調一次配分，扣分從 2.41 降到 0.140

**步驟**

1. 填 `.sum()` 做出 softmax，機率應該是 [0.665, 0.245, 0.090]；填電機的位置 2，扣分 2.408
2. 填 `backward`：門檻的梯度應該是 [0.665, 0.245, −0.910]，配分的梯度第一列是 [3.33, 1.33]
3. 填學習率 0.1，`opt.step()` 調一次之後，扣分變成 0.140

> 這題只調最後一層（W 和 c），跟投影片第 ⑤ 段的數字一樣；真的訓練會把整台機器的配分都交給 opt。

**要繳交**

- 截圖 ⑤：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ✏️ 練習五：機率、扣分、調配分（學長 C 讀電機）
h = torch.tensor([5., 2.])             # 文科分、理科分
W = W2.clone().requires_grad_()        # 最後一層的配分
c = b2.clone().requires_grad_()        # 最後一層的門檻
z = W @ h + c                          # 三個系的分數
p = torch.exp(z) / torch.exp(z).___()  # ✏️ 換成機率
loss = -torch.log(p[___])              # ✏️ 電機是第幾個
print('機率', p.detach(), '扣分', round(loss.item(), 3))
loss.___()                             # ✏️ 往回算責任
print('門檻的梯度', c.grad)
print('配分的梯度', W.grad)
opt = torch.optim.SGD([W, c], lr=___)  # ✏️ 學習率
opt.step()                             # 照責任調配分
print('調一次後', F.cross_entropy(W @ h + c, T).detach())

#### 📸 截圖 ⑤

跑完上面的 ✏️ 格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：機率與扣分、門檻的梯度、配分的梯度、調一次後的扣分

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. 門檻的梯度 [0.665, 0.245, −0.910] 和機率有什麼關係？哪一系的分數要往上調？配分的梯度第一列 [3.33, 1.33] 是怎麼來的？**

（在這裡寫）

**2. 把學習率改成 N÷100（N＝學號末兩碼，例如 07 → 0.07），調一次後的扣分是多少？比學習率 0.1 的 0.140 大還是小，為什麼？**

（在這裡寫）

## AI 使用紀錄

有用 AI 就寫下來，沒用就寫「沒有使用」。

| 用了哪個 AI | 問了什麼（大概就好） | 它哪裡幫到你 | 它哪裡說錯、你怎麼發現的 |
|---|---|---|---|
| （例：ChatGPT） | （例：W.T 是什麼意思） |  |  |

## 繳交前檢查

1. **重跑一次**：「執行階段 › 重新啟動工作階段並執行所有儲存格」，每一格都有輸出、沒有紅字。
2. **檔名**：`深度學習_W04上_學號_姓名.ipynb`（左上角點檔名就能改）。
3. **截圖**：5 張（①～⑤），每張都是整個螢幕，看得到身份指紋（左上的檔名、右上的 Google 頭貼、右下工作列的日期時間）和那一題的輸出。
4. **文字題**：五題的 ✏️ 文字回答都寫好，用到 N 的地方換成自己的學號末兩碼。
5. **開放分享**：右上角「分享」→「一般存取權」改成「知道連結的任何人」→ 角色選「檢視者」→「複製連結」。
6. **自己檢查**：開一個無痕視窗貼上連結，不用登入就打得開才算成功。
7. **下載**：「檔案 › 下載 › 下載 .ipynb」。
8. **交到 Flipclass**：分享連結 1 個、.ipynb 檔 1 個、截圖 5 張（①～⑤），交到 Flipclass 作業區，期限照 Flipclass 規定。